In [ ]:
# %pip install -q pyarrow

In [ ]:
import pandas as pd
from underthesea import sent_tokenize, word_tokenize
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Path Configuration

In [ ]:
input_path = "../data/raw/segmented_data.parquet"
cleaned_data_path = "../data/processed/cleaned_segmented_data.parquet"
cache_dir = "../data/cache/"

In [ ]:
df = pd.read_parquet(input_path, engine="pyarrow")

print(df.columns)
print(len(df))
df.head()

# Overview
The dataset has 3 columns: `title`, `url`, and `chunks`. Each row represents an article, with a list of text chunks derived from that article. The structure is as follows:

```text
- title: Str
- url: Str
- chunks: List[Str]
```

# Statistics

- Total number of articles
- Statistics on chunk and their distributions:
    - Chunk count per article.
    - Sentence count per chunk.
    - Word count per chunk.
    - Word count per sentence.
    - Approximate length in syllables/words.

In [ ]:
def get_statistics(df):
    
    # Chunk count per article
    chunk_counts = df['chunks'].apply(len)
    
    # Flatten all chunks for analysis
    all_chunks = [chunk for chunks in df['chunks'] for chunk in chunks]
    
    sentence_counts = [] # Sentences per chunk
    words_per_sentence = [] # Word count per sentence

    for chunk in all_chunks:
        sentences = sent_tokenize(chunk)
    
        # Sentence count per chunk
        sentence_counts.append(len(sentences))
        
        # Word count per sentence
        for sentence in sentences:
            words_per_sentence.append(len(word_tokenize(sentence)))

    words_per_chunk = [len(word_tokenize(chunk)) for chunk in all_chunks]
    
    return {
        'chunk_counts': chunk_counts,
        'sentence_counts': sentence_counts,
        'words_per_chunk': words_per_chunk,
        'words_per_sentence': words_per_sentence
    }
    
def visualize_distribution(stats: dict):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Chunk count per article
    sns.histplot(stats['chunk_counts'], ax=axes[0, 0], kde=True)
    axes[0, 0].set_title('Chunk Count per Article')
    axes[0, 0].set_xlabel('Number of Chunks')
    axes[0, 0].set_ylabel('Frequency')
    
    # Sentence count per chunk
    sns.histplot(stats['sentence_counts'], ax=axes[0, 1], kde=True)
    axes[0, 1].set_title('Sentence Count per Chunk')
    axes[0, 1].set_xlabel('Number of Sentences')
    axes[0, 1].set_ylabel('Frequency')
    
    # Word count per chunk
    sns.histplot(stats['words_per_chunk'], ax=axes[1, 0], kde=True)
    axes[1, 0].set_title('Word Count per Chunk')
    axes[1, 0].set_xlabel('Number of Words')
    axes[1, 0].set_ylabel('Frequency')
    
    # Word count per sentence
    sns.histplot(stats['words_per_sentence'], ax=axes[1, 1], kde=True)
    axes[1, 1].set_title('Word Count per Sentence')
    axes[1, 1].set_xlabel('Number of Words')
    axes[1, 1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    
def print_statistics(stats):
    """
    For each statistic, print mean, std, min, max and percentile (25th, 50th, 75th).
    """
    columns = ['chunk_counts', 'sentence_counts', 'words_per_chunk', 'words_per_sentence']
    for col in columns:
        mean = np.mean(stats[col])
        std = np.std(stats[col])
        min_val = np.min(stats[col])
        max_val = np.max(stats[col])
        p25 = np.percentile(stats[col], 25)
        p50 = np.percentile(stats[col], 50)
        p75 = np.percentile(stats[col], 75)
        print(f"===== {col} =====")
        print(f"Mean: {mean:.4f}")
        print(f"Standard Deviation: {std:.4f}")
        print(f"Min: {min_val}")
        print(f"Max: {max_val}")
        print(f"25th Percentile: {p25}")
        print(f"50th Percentile: {p50}")
        print(f"75th Percentile: {p75}")
        
    # visualize a boxplot for each statistic
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    sns.boxplot(x=stats['chunk_counts'], ax=axes[0, 0])
    axes[0, 0].set_title('Chunk Count per Article')
    sns.boxplot(x=stats['sentence_counts'], ax=axes[0, 1])
    axes[0, 1].set_title('Sentence Count per Chunk')
    sns.boxplot(x=stats['words_per_chunk'], ax=axes[1, 0])
    axes[1, 0].set_title('Word Count per Chunk')
    sns.boxplot(x=stats['words_per_sentence'], ax=axes[1, 1])
    axes[1, 1].set_title('Word Count per Sentence')
    plt.tight_layout()
    plt.show()

In [ ]:
# stats = get_statistics(df)

### Save stats and load

In [ ]:
import pickle

# with open(cache_dir + "eda_stats.pkl", "wb") as f:
#     pickle.dump(stats, f)

# print("Statistics saved to eda_stats.pkl")

with open(cache_dir + "eda_stats.pkl", "rb") as f:
    stats = pickle.load(f)

print("Statistics loaded from eda_stats.pkl")

print_statistics(stats)
visualize_distribution(stats)

# Data Quality & Cleaning

- Filter out articles that are too short or too long in words, sentences, or chunks.
- Clean formatting artifacts (extra whitespaces, special characters).
- Standardize text encoding (UTF-8).

In [ ]:
from tqdm import tqdm

def filter_articles(df, min_chunk_count, max_chunk_count, min_chunk_length, max_chunk_length):
    """
    Filter articles based on chunk count and chunk length criteria.
    """
    # Pre-filter by chunk count (fast, no tokenization needed)
    chunk_counts = df['chunks'].apply(len)
    mask = (chunk_counts >= min_chunk_count) & (chunk_counts <= max_chunk_count)
    candidates = df[mask]
    
    # Check word counts only for candidates that passed chunk count filter
    def check_chunk_lengths(chunks):
        for chunk in chunks:
            word_count = len(word_tokenize(chunk))
            if word_count < min_chunk_length or word_count > max_chunk_length:
                return False
        return True
    
    tqdm.pandas(desc="Filtering by chunk length")
    valid_mask = candidates['chunks'].progress_apply(check_chunk_lengths)
    
    filtered_df = candidates[valid_mask].reset_index(drop=True)
    
    print(f"Original articles: {len(df)}")
    print(f"Filtered articles: {len(filtered_df)}")
    print(f"Removed: {len(df) - len(filtered_df)} ({(len(df) - len(filtered_df)) / len(df) * 100:.2f}%)")
    
    return filtered_df

In [ ]:
min_chunk_length = 80
max_chunk_length = 350

min_chunk_count = 2
max_chunk_count = 6

filtered_df = filter_articles(df, min_chunk_count, max_chunk_count, min_chunk_length, max_chunk_length)

In [ ]:
filtered_df['chunks'] = filtered_df['chunks'].apply(lambda chunks: [c.strip() for c in chunks])
total_chunks = sum(len(chunks) for chunks in filtered_df['chunks'])
print(f"Total chunks after cleaning: {total_chunks}")

In [ ]:
filtered_df.to_parquet(cleaned_data_path, engine="pyarrow", index=False)
filtered_df.head()